# 07.03_Scanpy_Auco_ST_Analysis_Python

TACCO 空间映射、绘图与下游输入导出。

- 当前文件：`analysis/07_spatial_analysis/07.03_Scanpy_Auco_ST_Analysis_Python.ipynb`
- 原始来源：`Codes/07.03_Scanpy_Auco_ST_Analysis.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`anndata`, `json`, `matplotlib.pyplot`, `numpy`, `os`, `pandas`, `scanpy`, `seaborn`, `tacco`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


In [ ]:
fig_dir = '/share/home/zhangze/zz/NeuralOrigin/Figures'

## 0.Statistics

ZZ：45绘制小提琴统计图

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

st_AureliaMargin = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/04.SpatialTranscriptomicsProcessing/D06050D2.bin50_1.0.h5ad")

fig, axes = plt.subplots(1, 2, figsize=(10, 6))

# Max:3791, Q1:61, Q2:135, Q3:270, Min:1
sns.violinplot(
    data=st_AureliaMargin.obs,
    y="total_counts",
    inner="box",
    ax=axes[0],
    color="#5192DB"
)
axes[0].set_title("Univariate distribution of MiD Count")

# Max:948, Q1:33, Q2:68, Q3:128, Min:1
sns.violinplot(
    data=st_AureliaMargin.obs,
    y="n_genes_by_counts",
    inner="box",
    ax=axes[1],
    color="#92DCC6"
)
axes[1].set_title("Univariate distribution of Gene Type")

plt.tight_layout()

# output_path = "./Figures/01.Aurelia_violin_statistics.pdf"
fig.savefig(fig_dir+"/45.VlnPlot.Auco_ST_statistics.pdf", dpi=300, bbox_inches="tight")
fig.savefig(fig_dir+"/45.VlnPlot.Auco_ST_statistics.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
st_AureliaMargin

## 1.TACCO Analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import tacco as tc

In [ ]:
# 单细胞
adata_Auco_annotated = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellAnnotation/Auco.normalized.annotated.h5ad")
adata_Auco_annotated

In [ ]:
adata_Auco_annotated.X = adata_Auco_annotated.layers["counts"].copy()

In [ ]:
adata_Auco_annotated.X.toarray()

In [ ]:
adata_Auco_annotated.var.index.name = "FeatureID"
adata_Auco_annotated.var

In [ ]:
adata_Auco_annotated.obs['Broad_cell_type'].value_counts()

In [ ]:
# sub_map_str = {
#     'Auco_0':'EM_1',
#     'Auco_1':'EM_2',
#     'Auco_2':'EM_3',

#     'Auco_3':'GA_1',
#     'Auco_5':'GA_2',
#     'Auco_12':'GA_3',
#     'Auco_13':'GA_4',
#     'Auco_21':'GA_5',

#     'Auco_7':'NE_1',
#     'Auco_8':'NE_2',
#     'Auco_9':'NE_3',
#     'Auco_15':'NE_4',

#     'Auco_4':'CN_1',
#     'Auco_11':'CN_2',
#     'Auco_16':'CN_3',
#     'Auco_18':'CN_4',

#     'Auco_14':'GL_1',
#     'Auco_19':'GL_2',
#     'Auco_23':'GL_3',

#     'Auco_10':'SG_1',
#     'Auco_17':'SG_2',
#     'Auco_20':'SG_3',
#     'Auco_22':'SG_4',

#     'Auco_6':'HA',
# }

In [ ]:
selected_types = ['EM, Epidermal/Muscle cell', 'NE, Neural cell', 'CN, Cnidocytes/Nematocyte cell', 'HA, Hair cell']  # 替换为你的实际类型
# selected_types = ['EM_1', 'EM_2', 'EM_3', 
#                   'NE_1', 'NE_2', 'NE_3', 'NE_4',
#                   'CN_1', 'CN_2', 'CN_3', 'CN_4',
#                   'HA']  # 替换为你的实际类型
adata_Auco_annotated_filtered = adata_Auco_annotated[adata_Auco_annotated.obs['Broad_cell_type'].isin(selected_types)].copy()
# adata_Auco_annotated_filtered = adata_Auco_annotated[adata_Auco_annotated.obs['Sub_cell_type'].isin(selected_types)].copy()
adata_Auco_annotated_filtered.obs['Broad_cell_type'].value_counts()
# adata_Auco_annotated_filtered.obs['Sub_cell_type'].value_counts()

In [ ]:
# 空间组
adata_D2 = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/04.SpatialTranscriptomicsProcessing/D2_bin50.h5ad")
adata_D2

In [ ]:
adata_D2.X.toarray()

In [ ]:
# 3. 检查当前数据是否可能是counts
print(f"\n当前数据:")
print(f"X 数据类型: {type(adata_D2.X)}")
print(f"X 最小值: {adata_D2.X.min():.4f}")
print(f"X 最大值: {adata_D2.X.max():.4f}")
# print(f"X 是否包含小数: {any(adata_D2.X[0,:10] % 1 != 0)}")  # 检查是否为整数

In [ ]:
adata_D2.var['raw_var_names'] = adata_D2.var_names
adata_D2.var_names = adata_D2.var['real_gene_name']
# 使用rename方法修改var_names
adata_D2.raw.var.rename(index=dict(zip(adata_D2.raw.var_names, 
                                       adata_D2.var['real_gene_name'].values)), 
                       inplace=True)
adata_D2.var.index.name = "FeatureID"
adata_D2.var

In [ ]:
adata_D2.var_names

In [ ]:
adata_D2.var.index.to_series().to_csv(
    "/share/home/zhangze/zz/NeuralOrigin/Data/04.SpatialTranscriptomicsProcessing/TACCO_D2/D2_bin50.genes.txt",
    index=False,
    header=False
)

In [ ]:
# sc.pp.filter_cells(adata_D2, min_genes=80)
# sc.pp.filter_genes(adata_D2, min_cells=10)

In [ ]:
# ------------------------------
# 1. 加载单细胞与空间组数据
# ------------------------------
adata_sc = adata_Auco_annotated_filtered.copy()
adata_sp = adata_D2.copy()

print("Single-cell data:", adata_sc)
print("Spatial data:", adata_sp)

In [ ]:
# ------------------------------
# 2. 准备输入
# ------------------------------
# 假设单细胞的细胞类型标签在 adata_sc.obs['celltype'] 或 ['CellType'] 中
celltype_key = 'Broad_cell_type'
# celltype_key = 'Sub_cell_type'

# 检查表达矩阵一致性
shared_genes = list(set(adata_sc.var_names) & set(adata_sp.var_names))
print(f"共有基因 {len(shared_genes)} 个可用于匹配")

adata_sc = adata_sc[:, shared_genes].copy()
adata_sp = adata_sp[:, shared_genes].copy()

In [ ]:
# ------------------------------
# 3. 计算单细胞中各细胞类型的先验概率
# ------------------------------
prob = adata_sc.obs[celltype_key].value_counts() / adata_sc.n_obs
print("细胞类型先验分布：")
print(prob)

In [ ]:
# ------------------------------
# 4. 准备空间组结构
# ------------------------------
if 'X_spatial' not in adata_sp.obsm:
    adata_sp.obsm['X_spatial'] = adata_sp.obsm['spatial']

adata_sp.layers['data'] = adata_sp.X.copy()
adata_sp.X = adata_sp.layers['data']

In [ ]:
# ------------------------------
# 5. 运行 TACCO 映射注释
# ------------------------------
outpath = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/ST_annotated/Tacco_Auco_D2"
os.makedirs(outpath, exist_ok=True)

# 确保矩阵为浮点型
if not np.issubdtype(adata_sc.X.dtype, np.floating):
    adata_sc.X = adata_sc.X.astype(float)
if not np.issubdtype(adata_sp.X.dtype, np.floating):
    adata_sp.X = adata_sp.X.astype(float)


adata_sp = tc.tl.annotate(
    adata_sp,
    adata_sc,
    annotation_key=celltype_key,
    result_key=f'pred_{celltype_key}',
    annotation_prior=prob,
    verbose=True,
    # assume_valid_counts=True  # ✅ 允许非整数输入
)


In [ ]:
# # ------------------------------
# # 6. 保存预测结果
# # ------------------------------
# pred = adata_sp.obsm[f'pred_{celltype_key}'].copy()
# pred[f'pred_{celltype_key}'] = pred.idxmax(1)
# pred[f'pred_{celltype_key}_score'] = pred.max(1)

# adata_sp.obs[f'pred_{celltype_key}'] = pred[f'pred_{celltype_key}']
# adata_sp.obs[f'pred_{celltype_key}_score'] = pred[f'pred_{celltype_key}_score']

# pred.to_csv(os.path.join(outpath, f"D2_pred_{celltype_key}.csv.gz"))
# adata_sp.write(os.path.join(outpath, "D2_mapped.h5ad"))

# ------------------------------
# 6. 保存预测结果（修正版）
# ------------------------------
pred = adata_sp.obsm[f'pred_{celltype_key}'].copy()

# ✅ 只保留数值列（忽略 Broad_cell_type / pred_Broad_cell_type）
numeric_cols = pred.select_dtypes(include=[np.number]).columns
pred_numeric = pred[numeric_cols]

# ✅ 计算最大概率对应的细胞类型与分数
pred[f'pred_{celltype_key}'] = pred_numeric.idxmax(1)
pred[f'pred_{celltype_key}_score'] = pred_numeric.max(1)

# 写回到 obs
adata_sp.obs[f'pred_{celltype_key}'] = pred[f'pred_{celltype_key}']
adata_sp.obs[f'pred_{celltype_key}_score'] = pred[f'pred_{celltype_key}_score']

# 保存结果
outpath = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/ST_annotated/Tacco_Auco_D2"
os.makedirs(outpath, exist_ok=True)
pred.to_csv(os.path.join(outpath, f"D2_pred_{celltype_key}.csv"))
adata_sp.write(os.path.join(outpath, "D2_mapped.h5ad"))

## 2.UMAP Plot

In [ ]:
print(pred_numeric.columns)

In [ ]:
pred[['pred_Broad_cell_type', 'pred_Broad_cell_type_score']].head()
# pred[['pred_Sub_cell_type', 'pred_Sub_cell_type_score']].head()

In [ ]:
# ------------------------------
# 7. 绘图展示
# ------------------------------
sc.settings.figdir = outpath

# 总体预测分布
sc.pl.embedding(
    adata_sp,
    basis='X_spatial',
    # color=f'pred_{celltype_key}',
    color='BroadType',
    title='Spatial Bins',
    s=3,
    save="_predicted_types.png"
)
# sc.pl.spatial(
#     adata_sp,
#     color=f'pred_{celltype_key}',
#     spot_size=1,
#     title='Predicted cell types',
#     save="_predicted_types.png"
# )

In [ ]:
# 绑定颜色
BroadType_colors = {
    'Cnidocytes': '#17becf', # CN 青色 '#17becf'
    'Epidermal/Muscle': '#9467bd',      # EM 紫色 '#9467bd'
    'Sensory': '#8c564b',                  # HA 棕色 '#8c564b'
    'Neural-related': '#2ca02c'               # NE 绿色 '#2ca02c'
}

sc.pl.embedding(
    adata_sp,
    basis='X_spatial',
    # color=f'pred_{celltype_key}',
    color='BroadType',
    palette=BroadType_colors,  # 使用palette参数传入颜色字典
    title='Spatial Bins',
    s=3,
    save="_predicted_types.png"
)

ZZ：46绘制空间图谱

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 绑定颜色
BroadType_colors = {
    'Cnidocytes': '#17becf', # CN 青色 '#17becf'
    'Epidermal/Muscle': '#9467bd',      # EM 紫色 '#9467bd'
    'Sensory': '#8c564b',                  # HA 棕色 '#8c564b'
    'Neural-related': '#2ca02c'               # NE 绿色 '#2ca02c'
}
# st_AureliaMargin = sc.read_h5ad("./Figures/AureliaMargin.annos.h5ad")

fig, ax = plt.subplots(dpi=300)

sc.pl.spatial(
    adata_sp,
    color="BroadType",
    palette=BroadType_colors,  # 使用palette参数传入颜色字典
    spot_size=50,
    img=None,           # ⭐ 不显示组织图，只显示点
    legend_loc=None,    # ⭐ 不显示图例（可选）
    frameon=False,      # ⭐ 去掉边框
    ax=ax,
    show=False
)

# # 黑色背景
ax.set_facecolor("black")
fig.patch.set_facecolor("black")

# 去掉坐标轴标签
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

fig.savefig(fig_dir+"/46.Scatter.Auco_ST_black.png", dpi=300, bbox_inches="tight")
fig.savefig(fig_dir+"/46.Scatter.Auco_ST_black.pdf", dpi=300, bbox_inches="tight")
plt.show()


ZZ：47绘制空间图谱(透明底)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# st_AureliaMargin = sc.read_h5ad("./Figures/AureliaMargin.annos.h5ad")

fig, ax = plt.subplots(dpi=300)

sc.pl.spatial(
    adata_sp,
    color="BroadType",
    spot_size=50,
    img=None,           # ⭐ 不显示组织图，只显示点
    legend_loc=None,    # ⭐ 不显示图例（可选）
    frameon=False,      # ⭐ 去掉边框
    ax=ax,
    show=False
)

# ⭐ 设置透明背景
ax.set_facecolor("none")  # 设置坐标轴背景为透明
fig.patch.set_alpha(0)     # 设置图形背景透明度为0（完全透明）

# 去掉坐标轴标签
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

fig.savefig(fig_dir+"/47.Scatter.Auco_ST_white.png", dpi=300, bbox_inches="tight", transparent=True)
fig.savefig(fig_dir+"/47.Scatter.Auco_ST_white.pdf", dpi=300, bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
# each cluster in super_leiden
celltypes = adata_sp.obs['BroadType'].unique().tolist()

for ct in celltypes:
    sc.pl.spatial(
        adata_sp,
        color='BroadType',
        groups=[ct],
        spot_size=100,
        title=f"Spatial map: {ct}"
    )


ZZ：48绘制空间图谱

In [ ]:
adata_sp.obs['BroadType'].unique().tolist()

In [ ]:
# each cluster in super_leiden
# celltypes = adata_sp.obs['BroadType'].unique().tolist()
celltypes = ['Sensory', 'Neural-related', 'Epidermal/Muscle', 'Cnidocytes']

for ct in celltypes:

    fig, ax = plt.subplots(dpi=300)
    sc.pl.spatial(
        adata_sp,
        color="BroadType",
        groups=[ct],
        spot_size=50,
        img=None,           # ⭐ 不显示组织图，只显示点
        legend_loc=None,    # ⭐ 不显示图例（可选）
        frameon=False,      # ⭐ 去掉边框
        ax=ax,
        show=False
    )

    # ⭐ 设置透明背景
    ax.set_facecolor("none")  # 设置坐标轴背景为透明
    fig.patch.set_alpha(0)     # 设置图形背景透明度为0（完全透明）

    # 去掉坐标轴标签
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")

    fig.savefig(fig_dir+"/48.Scatter.Auco_ST_"+ct[0:5]+"_white.png", dpi=300, bbox_inches="tight", transparent=True)
    fig.savefig(fig_dir+"/48.Scatter.Auco_ST_"+ct[0:5]+"_white.pdf", dpi=300, bbox_inches="tight", transparent=True)
    plt.show()

## 3. Markers

In [ ]:
import scanpy as sc

# 假设 marker_list 是你想要绘制的标记基因列表
marker_list = [
    # "CN, Cnidocytes/Nematocyte cell": 
    *[
        # 'XLOC-010665#MYC-MARMO#P22555'
        'XLOC-021328#XLOC-021328',
        'XLOC-010138#MYC-PONPY#A2T7L5',
        'XLOC-010662#MYC-ASTRU#Q17103',
        'gene-evm.model.ptg000016l.438#SYT1-CAEEL#P34693',
        'gene-evm.model.ptg000016l.445#MYC-PONPY#A2T7L5',
        'gene-evm.model.ptg000016l.468#MYC-ASTRU#Q17103',
        'gene-evm.model.ptg000049l.4-1#MYC-MARMO#P22555',
    ],
    # "Epidermal/Muscle": 
    *[
        # 'XLOC-011330#CALM-MACPY#Q40302',
        # 'XLOC-004173#DD3-DICDI#Q58A42',
        # 'XLOC-026552#MLE-BRAFL#Q17133',
        # 'XLOC-014937#SVIL-BOVIN#O46385',
        # 'XLOC-016636#MYL6B-HUMAN#P14649',
        # 'XLOC-018113#MYL6B-HUMAN#P14649',
        'XLOC-026007#TPM1-PODCA#P41114',
        'XLOC-024876#TPM2-PODCA#Q9U5M4',
        'XLOC-003006#APLP-LOCMI#Q9U943',
        # 'gene-evm.model.ptg000006l.268#MYL1-DANRE#Q6P0G6',
        # 'XLOC-016635#MYL1-DANRE#Q6P0G6',
        # 'gene-evm.model.ptg000002l.132#MLE-BRAFL#Q17133'
    ],
    # "Sensory": 
    *[
        'XLOC-001822#PKD2-BOVIN#Q4GZT3',
        # 'XLOC-015184#CAC1E-MOUSE#Q61290',
        # 'gene-evm.model.ptg000003l.332#TBA1A-CHICK#P02552',
        'XLOC-006274#AVIL-RAT#Q9WU06'
    ],
    # "Neural-related": 
    *[
        'XLOC-000171#TBA1A-CHICK#P02552',
        'XLOC-009030#TBA-LYTPI#P02553',
        'XLOC-009501#TBA1A-CHICK#P02552',
        'XLOC-013650#TBA1-PARLI#P18258',
        'gene-evm.model.ptg000003l.332#TBA1A-CHICK#P02552',
        'gene-evm.model.ptg000024l.725#TBA1-PARLI#P18258',
        'gene-evm.model.ptg000024l.726#TBA1A-CHICK#P02552',
        'XLOC-019965#SYT14-HUMAN#Q8NB59',
        'XLOC-009553#SNX27-MOUSE#Q3UHD6',
        'XLOC-013940#HCN2-HUMAN#Q9UL51',
        'XLOC-022104#CNGA2-BOVIN#Q03041',
        'XLOC-024044#CNGA3-MOUSE#Q9JJZ8',
        'XLOC-025648#HCN4-HUMAN#Q9Y3Q4'
        # 'XLOC-006274#AVIL-RAT#Q9WU06'
        # 'gene-evm.model.ptg000013l.805#CAB32-DROME#P41044',
        # 'gene-evm.model.ptg000022l.585#ACH1-CAEEL#P48180',
        # 'gene-evm.model.ptg000001l.686#SOX14-DANRE#Q32PP9'
    ]
]

# sc.settings.figdir = fig_dir
# sc.settings.set_figure_params(dpi=300, dpi_save=300)

# 只保留数据中存在的基因
available_genes = adata_sp.var_names.tolist()
marker_list_filtered = [gene for gene in marker_list if gene in available_genes]

# 检查被过滤掉的基因
missing_genes = [gene for gene in marker_list if gene not in available_genes]
if missing_genes:
    print(f"以下基因不存在于数据中，将被过滤掉：")
    for gene in missing_genes:
        print(f"  - {gene}")
    print(f"\n原始marker数量: {len(marker_list)}")
    print(f"过滤后marker数量: {len(marker_list_filtered)}")

# 使用 DotPlot 来绘制每个 broad_cell_types 分组下的基因表达
sc.pl.dotplot(
    adata_sp, 
    var_names=marker_list_filtered,  # marker 基因
    groupby='BroadType',  # 根据 broad_cell_types 分组
    # use_raw=False,  # 如果你使用的是 raw 数据集，可以设置为 True
    dot_max=1,  # 控制最大点大小
    dot_min=0,  # 控制最小点大小
    standard_scale='var',  # 选项 ['var', 'obs']，是否按基因或细胞标准化
    # color_map='viridis',  # 设置颜色映射
    swap_axes=True,
    show=True,  # 设置为 True 显示图像
    figsize=(8, 6),
    # save="43.DotPlot.BroadType_markers.pdf"
)

In [ ]:
markers_dict = {
    "CN, Cnidocytes/Nematocyte cell": [
        # 'XLOC-010665#MYC-MARMO#P22555'
        'XLOC-021328#XLOC-021328',
        # 'XLOC-010138#MYC-PONPY#A2T7L5',
        # 'XLOC-010662#MYC-ASTRU#Q17103',
        'gene-evm.model.ptg000016l.438#SYT1-CAEEL#P34693'
        # 'gene-evm.model.ptg000016l.445#MYC-PONPY#A2T7L5',
        # 'gene-evm.model.ptg000016l.468#MYC-ASTRU#Q17103',
        # 'gene-evm.model.ptg000049l.4-1#MYC-MARMO#P22555',
    ],
    "Epidermal/Muscle": [
        # 'XLOC-011330#CALM-MACPY#Q40302',
        # 'XLOC-004173#DD3-DICDI#Q58A42',
        # 'XLOC-026552#MLE-BRAFL#Q17133',
        # 'XLOC-014937#SVIL-BOVIN#O46385',
        # 'XLOC-016636#MYL6B-HUMAN#P14649',
        # 'XLOC-018113#MYL6B-HUMAN#P14649',
        'XLOC-026007#TPM1-PODCA#P41114',
        'XLOC-024876#TPM2-PODCA#Q9U5M4',
        # 'XLOC-003006#APLP-LOCMI#Q9U943'
        # 'gene-evm.model.ptg000006l.268#MYL1-DANRE#Q6P0G6',
        # 'XLOC-016635#MYL1-DANRE#Q6P0G6',
        # 'gene-evm.model.ptg000002l.132#MLE-BRAFL#Q17133'
    ],
    "Sensory": [
        'XLOC-001822#PKD2-BOVIN#Q4GZT3',
        # 'XLOC-015184#CAC1E-MOUSE#Q61290',
        # 'gene-evm.model.ptg000003l.332#TBA1A-CHICK#P02552',
        'XLOC-006274#AVIL-RAT#Q9WU06'
    ],
    "Neural-related": [
        # 'XLOC-000171#TBA1A-CHICK#P02552',
        # 'XLOC-009030#TBA-LYTPI#P02553',
        # 'XLOC-009501#TBA1A-CHICK#P02552',
        'XLOC-013650#TBA1-PARLI#P18258',
        # 'gene-evm.model.ptg000003l.332#TBA1A-CHICK#P02552',
        # 'gene-evm.model.ptg000024l.725#TBA1-PARLI#P18258',
        # 'gene-evm.model.ptg000024l.726#TBA1A-CHICK#P02552',
        'XLOC-019965#SYT14-HUMAN#Q8NB59',
        # 'XLOC-009553#SNX27-MOUSE#Q3UHD6',
        # 'XLOC-013940#HCN2-HUMAN#Q9UL51',
        # 'XLOC-022104#CNGA2-BOVIN#Q03041',
        # 'XLOC-024044#CNGA3-MOUSE#Q9JJZ8',
        # 'XLOC-025648#HCN4-HUMAN#Q9Y3Q4'
        # 'XLOC-006274#AVIL-RAT#Q9WU06'
        # 'gene-evm.model.ptg000013l.805#CAB32-DROME#P41044',
        # 'gene-evm.model.ptg000022l.585#ACH1-CAEEL#P48180',
        # 'gene-evm.model.ptg000001l.686#SOX14-DANRE#Q32PP9'
    ]
}

In [ ]:
# Loop through cell-type marker groups and plot violins
for celltype, marker_feature_list in markers_dict.items():
    print(f"\n=== {celltype} markers ===")

    # --- AureliaMargin-ST ---
    axs = sc.pl.violin(
        adata_sp,
        keys=marker_feature_list,
        groupby='BroadType',
        jitter=0.4,
        rotation=45,
        stripplot=False,
        multi_panel=True,
        show=False      # ⭐ allow modifying axes before display
    )
    fig = axs[0].figure  # 获取整个图形
    fig.savefig(fig_dir+"/49.VlnPlot.Auco_ST_"+celltype[0:5]+".pdf", dpi=300)
    # When multi_panel=True, axs is a list of Axes
    for ax in axs:
        ax.set_xlabel("Auco (ST)")
    plt.show()
    plt.close()

    # --- ClytiaMedusa-SC ---
    axs = sc.pl.violin(
        adata_Auco_annotated,
        keys=marker_feature_list,
        groupby=celltype_key,
        jitter=0.4,
        rotation=45,
        stripplot=False,
        multi_panel=True,
        show=False
    )
    fig = axs[0].figure  # 获取整个图形
    fig.savefig(fig_dir+"/50.VlnPlot.Auco_SC_"+celltype[0:5]+".pdf", dpi=300)
    for ax in axs:
        ax.set_xlabel("Auco (SC)")
    plt.show()
    plt.close()

## 4.Orthogroup Processing

### 4.1 transcripts to OGs mapping

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad


def map_transcripts_to_orthogroups(
    adata: ad.AnnData,
    og_map_path: str,
    save_path: str = None,
    agg_method: str = "sum"
) -> ad.AnnData:
    """
    Map transcript-level expression in an AnnData object to orthogroups (OGs),
    aggregate expression across transcripts belonging to the same OG, and
    return a new AnnData object at the OG level.

    Parameters
    ----------
    adata : AnnData
        Input AnnData object containing transcript-level expression.
    og_map_path : str
        Path to a CSV file containing two columns:
        `protein_id` and `orthogroup`, defining transcript-to-OG mapping.
    save_path : str, optional
        If provided, the output OG-level AnnData will be written to this path.
    agg_method : {"sum", "mean"}, optional
        Aggregation method for transcripts belonging to the same OG.
        Default is "sum".

    Returns
    -------
    AnnData
        An AnnData object with expression aggregated per orthogroup.
    """

    print("\n===== Mapping transcripts to orthogroups =====")
    print(f"Input AnnData: {adata}")
    adata = adata.copy()  # avoid modifying the original object

    # -------------------------------
    # 1. Report input dimension
    # -------------------------------
    print(f"Number of transcripts (vars) before mapping: {adata.n_vars}")

    # Store original gene/transcript IDs
    adata.var["raw_gene_id"] = adata.var_names

    # -------------------------------
    # 2. Load OG mapping table
    # -------------------------------
    og_df = pd.read_csv(og_map_path)
    if not {"protein_id", "orthogroup"}.issubset(og_df.columns):
        raise ValueError("OG map must contain columns: 'protein_id', 'orthogroup'.")

    og_map = og_df.set_index("protein_id")["orthogroup"].to_dict()
    unique_og_count = len(set(og_map.values()))

    print(f"OG mapping table loaded: {unique_og_count} unique OGs, {len(og_map)} total proteins.")

    # -------------------------------
    # 3. Build orthogroup column
    # -------------------------------
    adata.var["orthogroup"] = adata.var_names.map(lambda x: og_map.get(x, x))

    n_mapped = np.sum(adata.var["orthogroup"] != adata.var["raw_gene_id"])
    n_unmapped = np.sum(adata.var["orthogroup"] == adata.var["raw_gene_id"])

    print(f"Transcripts mapped to OGs: {n_mapped}")
    print(f"Transcripts not mapped (kept as unique IDs): {n_unmapped}")

    # -------------------------------
    # 4. Construct expression DataFrame
    # -------------------------------
    # Convert sparse matrix if necessary
    X = adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X

    expr_df = pd.DataFrame(
        X,
        index=adata.obs_names,
        columns=adata.var["orthogroup"].values,  # renamed to OG or original
    )

    og_dim_before = expr_df.shape[1]

    # -------------------------------
    # 5. Aggregate expression per OG
    # -------------------------------
    if agg_method == "sum":
        expr_og = expr_df.groupby(expr_df.columns, axis=1).sum()
    elif agg_method == "mean":
        expr_og = expr_df.groupby(expr_df.columns, axis=1).mean()
    else:
        raise ValueError("agg_method must be 'sum' or 'mean'.")

    og_dim_after = expr_og.shape[1]

    print(f"Expression matrix before aggregation: {og_dim_before} columns (with duplicates).")
    print(f"Expression matrix after aggregation:  {og_dim_after} unique OGs.")

    # -------------------------------
    # 6. Build new AnnData object
    # -------------------------------
    adata_og = ad.AnnData(
        X=expr_og.values,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=expr_og.columns)
    )

    # Transfer metadata
    adata_og.uns = adata.uns.copy()
    adata_og.obsm = adata.obsm.copy()

    print(f"Output AnnData: cells = {adata_og.n_obs}, OGs = {adata_og.n_vars}")

    # -------------------------------
    # 7. Save if requested
    # -------------------------------
    if save_path is not None:
        adata_og.write_h5ad(save_path, compression="gzip")
        print(f"OG-level AnnData saved to: {save_path}")

    print("===== Mapping completed. =====\n")
    return adata_og

In [ ]:
og_map_path = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/Auco.protein_to_orthogroup.csv"

### 4.2 Auco ST mapped

In [ ]:
# ST
st_Auco_og = map_transcripts_to_orthogroups(adata_sp, og_map_path)
print(st_Auco_og)

In [ ]:
st_Auco_og.obs.index.name = "CellName"
st_Auco_og.obs

In [ ]:
st_Auco_og.var.index.name = "OrthoGene"
st_Auco_og.var

In [ ]:
# save Aurelia ST OGs adata
raw_st_Auco_og_path = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_sc_st/Auco.bin50.OGs.normalized.h5ad"
st_Auco_og.write(raw_st_Auco_og_path)

In [ ]:
# save Aurelia ST OGs
st_Auco_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_sc_st/Auco.bin50.OGs.txt', index=False, header=False)

### 4.3 Auco SC mapped

In [ ]:
# SC
sc_Auco_og = map_transcripts_to_orthogroups(adata_Auco_annotated, og_map_path)
print(sc_Auco_og)

In [ ]:
sc_Auco_og.obs.index.name = "CellName"
sc_Auco_og.obs

In [ ]:
sc_Auco_og.var.index.name = "OrthoGene"
sc_Auco_og.var

In [ ]:
sc.pl.umap(sc_Auco_og, color=['Broad_cell_type'])

In [ ]:
# save Aurelia SC OGs adata
raw_sc_Auco_og_path = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_sc_st/Auco.sc.OGs.normalized.h5ad"
sc_Auco_og.write(raw_sc_Auco_og_path, compression="gzip")

In [ ]:
# save Aurelia SC OGs
sc_Auco_og.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_sc_st/Auco.sc.OGs.txt', index=False, header=False)

## 5. Save exp.txt and metadata.txt for MDIC3

### 5.1 导出ST

In [ ]:
adata = adata_sp.copy()
adata

In [ ]:
sc.pp.filter_cells(adata, min_genes=20)
sc.pp.filter_genes(adata, min_cells=3)
adata

In [ ]:
adata.obs['BroadType'].value_counts()

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=5000,
    flavor="seurat_v3"
)

adata_md = adata[:, adata.var["highly_variable"]].copy()
print(adata_md.shape)  # 7146 × ~5000

In [ ]:
adata_md

In [ ]:
if not isinstance(adata_md.X, np.ndarray):
    expr = adata_md.X.toarray()
else:
    expr = adata_md.X

expr_df = pd.DataFrame(
    expr,
    index=adata_md.obs_names,
    columns=adata_md.var_names
)

expr_df = expr_df.T

print(expr_df.shape)
expr_df.head()

In [ ]:
exp_path = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_ST.exp.txt"

with open(exp_path, "w") as f:
    f.write("\t" + "\t".join(expr_df.columns) + "\n")
    for gene, values in expr_df.iterrows():
        f.write(gene + "\t" + "\t".join(map(str, values.values)) + "\n")

In [ ]:
meta_df = adata_md.obs.loc[expr_df.columns, ["BroadType"]]
meta_df["cell"] = meta_df.index

meta_df = meta_df[["cell", "BroadType"]]

meta_path = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_ST.metadata.txt"
meta_df.to_csv(meta_path, sep="\t", header=False, index=False)

### 5.2 导出SC

In [ ]:
adata = adata_Auco_annotated.copy()
adata

In [ ]:
sc.pp.filter_cells(adata, min_genes=20)
sc.pp.filter_genes(adata, min_cells=3)
adata

In [ ]:
adata.obs['Broad_cell_type'].value_counts()

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=5000,
    flavor="seurat_v3"
)

adata_md = adata[:, adata.var["highly_variable"]].copy()
print(adata_md.shape)  # 7146 × ~5000

In [ ]:
adata_md

In [ ]:
if not isinstance(adata_md.X, np.ndarray):
    expr = adata_md.X.toarray()
else:
    expr = adata_md.X

expr_df = pd.DataFrame(
    expr,
    index=adata_md.obs_names,
    columns=adata_md.var_names
)

expr_df = expr_df.T

print(expr_df.shape)
expr_df.head()

In [ ]:
exp_path = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.exp.txt"

with open(exp_path, "w") as f:
    f.write("\t" + "\t".join(expr_df.columns) + "\n")
    for gene, values in expr_df.iterrows():
        f.write(gene + "\t" + "\t".join(map(str, values.values)) + "\n")

In [ ]:
meta_df = adata_md.obs.loc[expr_df.columns, ["Broad_cell_type"]]
meta_df["cell"] = meta_df.index

meta_df = meta_df[["cell", "Broad_cell_type"]]

meta_path = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/auco_MDIC3/Auco_SC.metadata.txt"
meta_df.to_csv(meta_path, sep="\t", header=False, index=False)

## 6.导出echarts json

In [ ]:
import scanpy as sc

adata_sp = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/ST_annotated/Tacco_Auco_D2/D2_mapped.h5ad")

map_str = {
    'EM, Epidermal/Muscle cell':'Epidermal/Muscle',
    'CN, Cnidocytes/Nematocyte cell':'Cnidocytes',
    'NE, Neural cell':'Neural-related',
    'HA, Hair cell':'Sensory'
}

ser = adata_sp.obs['pred_Broad_cell_type']
adata_sp.obs['BroadType'] = ser.astype(str).map(map_str)
adata_sp.obs

In [ ]:
adata_sp.obs['BroadType'].value_counts()

In [ ]:
import json
import numpy as np
import pandas as pd

def export_spatial_atlas_json(adata):
    # 定义输出路径
    json_path = "/share/home/zhangze/zz/NeuralOrigin/Figures/Aco_spatial_atlas.json"
    
    # 1. 准备类别顺序和配色（按你之前的定义）
    categories = ['Epidermal/Muscle', 'Cnidocytes', 'Sensory', 'Neural-related']
    palette = ['#9467bd', '#17becf', '#8c564b', '#2ca02c'] # 对应 EM, CN, HA, NE
    
    # 建立类别到数字索引的映射
    cat_to_idx = {cat: i for i, cat in enumerate(categories)}
    
    # 2. 提取坐标和映射类别索引
    # 确保 BroadType 是字符串类型以便映射
    adata.obs['BroadType'] = adata.obs['BroadType'].astype(str)
    
    # 提取 x, y 坐标和映射后的索引
    x_coords = adata.obs['x'].values.astype(float)
    y_coords = adata.obs['y'].values.astype(float)
    # 使用 map 映射索引，处理可能的空值
    indices = adata.obs['BroadType'].map(cat_to_idx).fillna(-1).values.astype(int)
    
    # 3. 组合成 ECharts scatter 识别的 [[x, y, type_idx], ...] 格式
    # 使用 numpy 堆叠速度最快
    simple_data = np.column_stack([x_coords, y_coords, indices]).tolist()
    
    # 4. 构建最终 JSON 结构
    output = {
        "metadata": {
            "title": "Aurelia coerulea Stereo-seq Spatial Atlas",
            "resolution": "bin50",
            "count": len(simple_data)
        },
        "categories": categories,
        "palette": palette,
        "data": simple_data
    }
    
    # 5. 写入文件
    with open(json_path, "w", encoding='utf-8') as f:
        json.dump(output, f)
    
    print(f"成功导出 {len(simple_data)} 个点至: {json_path}")

# 执行导出
export_spatial_atlas_json(adata_sp)